In [1]:
import sys
import os
from pathlib import Path

import sqlglot
import yaml

from src.core.ext_parser import ExtParser
from src.core.parser import HiveScriptParser
from src.transformers.raw_pyspark_transformer import RawPySparkTransformer
from src.jinja.environment import render_template
from src.paths import *

In [2]:

variable_path = VARIABLE_CONFIG_PATH

In [3]:
from utils.file_utils import parse_file_name

def get_paths(script_name: str):

    layer, sub_layer, source_name, base_table = parse_file_name(script_name)

    # script_name = "com_t_mhbos_m_client"
    datalake_type_subfolder = 'dml'
    datalake_layer_subfolder = script_name.split("_")[0]

    # sql_file_path = PROJECT_ROOT / "output" / "input" / "ddl" / "raw" / f"{script_name}", 
    sql_file_path = DATALAKE_SCRIPT_DIR / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}", 
    ext_file_path = DATALAKE_SCRIPT_DIR / "ext" / "xml" / f"{source_name}_{base_table}.xml"
    output_file_path = PROJECT_ROOT / "output" / "migration" / datalake_type_subfolder / datalake_layer_subfolder / f"{script_name}.py"

    return source_name, base_table, sql_file_path, ext_file_path, output_file_path

In [4]:
def generate_scripts(script_name: str):
    source_name, base_table, sql_file_path, ext_file_path, output_file_path = get_paths(script_name)

    print(f"[1] Parsing file {sql_file_path.name}...")
    context = HiveScriptParser.parse_file(str(sql_file_path))
    ext_parser = ExtParser()
    context.ext_context = ext_parser.parse_ext(ext_file_path)

    print("[2] Initializing Optimized Transformer...")
    transformer = RawPySparkTransformer(config_root=PROJECT_ROOT / "configs")
    render_model = transformer.transform(context)

    print("[3] Rendering Template (optimized_pyspark.jinja)...")
    final_script = render_template(
        # template_name="pyspark/optimized_pyspark.jinja",
        template_name="pyspark/raw_in_df_transform.jinja",
        render_model=render_model
    )

    # 2. Load mapping configuration from YAML
    output_file_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(final_script)

    print("[4] Rendering DAGs (optimized_pyspark.jinja)...")
    dag = render_template(
        # template_name="pyspark/optimized_pyspark.jinja",
        template_name="migration/dags/dag_daily.jinja",
        render_model=render_model
    )

    # 2. Load mapping configuration from YAML
    dag_output = PROJECT_ROOT / "output" / "migration" / "dags" / "erc" / f"dag_erc_{source_name}_{base_table}.py"
    dag_output.parent.mkdir(parents=True, exist_ok=True)
    with open(dag_output, 'w', encoding='utf-8') as f:
        f.write(dag)

    print("[5] Rendering DAGs (optimized_pyspark.jinja)...")
    dag = render_template(
        # template_name="pyspark/optimized_pyspark.jinja",
        template_name="migration/dags/dag_daily_nas.jinja",
        render_model=render_model
    )

    # 2. Load mapping configuration from YAML
    dag_output = PROJECT_ROOT / "output" / "migration" / "dags" / "erc" / f"dag_nas_erc_{source_name}_{base_table}.py"
    dag_output.parent.mkdir(parents=True, exist_ok=True)
    with open(dag_output, 'w', encoding='utf-8') as f:
        f.write(dag)


    # print("\n" + "=" * 50)
    # print("OPTIMIZED PYTHON FILE RESULT")
    # print("=" * 50 + "\n")
    # print(final_script)


In [14]:
table_list = [
    # "smfkibb_tbl_v3_cache_accountmarginposition",
    # "smfkibb_tbl_security",
    # "smfkibb_tbl_policytype",
    # "smfkibb_tbl_account",
    # "sblkibb_tbl_typeofbusiness",
    # "sblkibb_tbl_title",
    # "sblkibb_tbl_residencystatus",
    # "sblkibb_tbl_race",
    # "sblkibb_tbl_occupation",
    # "sblkibb_tbl_maritalstatus",
    # "sblkibb_tbl_counterpartyclass",
    # "sblkibb_tbl_counterparty",
    # "sblkibb_tbl_account",
    # "sbl_tbl_einvoicing_clientdata",
    # "m21_race",
    # "m21_occupation",
    # "m21_currency",
    # "m21_country",
    # "m21_clienttype",
    # "m21_bankaccount",
    # "m21_aetype"
    # "toms_eagentsb",
    # "toms_eagentsb_iuta",
    # "m21_o_trx",
    # "m21_a_trx",
    # "kdi_trx",
    # "guava_trx_inc",
    # "mhbos_ecmit029r",
    # "mhbos_ecmit028r",
    # "general_country",
    # "m21_natureofbusiness"

    # "m21_cashmovement",
    # "m21_countrystate",
    
#     "cur_dim_indicators_rak",
# "cur_dim_indicators_kdi",
# "cur_dim_indicators_mhbos",
# "cur_dim_indicators_lms",
# "cur_dim_indicators_m21",
# "cur_dim_indicators_toms_eretail",
# "cur_dim_indicators_toms_ecorporate",
# "cur_dim_indicators_sbl",

    

]

for table in table_list:
    try:
        generate_scripts(f"raw_{table}")
    except:
        generate_scripts(table)


[1] Parsing file raw_cur_dim_indicators_rak.sql...
[1] Parsing file cur_dim_indicators_rak.sql...


FileNotFoundError: The file C:\Users\ext_giadung\projects\datalake-script\ext\xml\dim_indicators.xml does not exist.

In [ ]:
'smfkibb_tbl_v3_cache_accountmarginposition'.split('_', 1)

In [11]:

for table in table_list:
    print(
    f"""
INSERT INTO public.c_flow_config_ds (flow_name,schedule,email,email_on_failure,email_on_retry,retries,retry_delay,catchup) VALUES
     ('erc_{table}','None','datalakedevteam@kenanga.com.my','False','False','1','5','False');
    """
    )


INSERT INTO public.c_flow_config (flow_name,schedule,email,email_on_failure,email_on_retry,retries,retry_delay,catchup) VALUES
     ('erc_m21_countrystate','None','datalakedevteam@kenanga.com.my','False','False','1','5','False');
    


In [12]:
for table in table_list:
    source = table.split('_', 1)[0]
    table_name = table.split('_', 1)[1]
    print(f"""
INSERT INTO public.c_etl_run (source_name,table_name,last_etl_start_time,last_etl_end_time,last_ext_start_time,last_ext_end_time,ext_start_time,ext_end_time) VALUES
	 ('{source}','{table_name}','2026-06-18 09:25:07.616',NULL,'2026-06-16 09:25:07','2026-06-17 09:25:07','2026-06-10 09:25:07','2026-06-15 09:25:07');
"""
)


INSERT INTO public.c_etl_run (source_name,table_name,last_etl_start_time,last_etl_end_time,last_ext_start_time,last_ext_end_time,ext_start_time,ext_end_time) VALUES
	 ('m21','countrystate','2026-06-18 09:25:07.616',NULL,'2026-06-16 09:25:07','2026-06-17 09:25:07','2026-06-10 09:25:07','2026-06-15 09:25:07');



In [ ]:
for table in table_list:
    ddl_file_path = DATALAKE_SCRIPT_DIR / "ddl" / "raw" / f"raw_{table}", 
    print(ddl_file_path.read_text())